# Visualization — earthlens fetches, digitalearth draws

earthlens owns no plotting layer. It fetches pixels from Earth Engine and returns
pyramids objects; every figure, map, animation and chart is produced by
[**digitalearth**](https://github.com/serapeum-org/Digital-Earth) over cleopatra,
across its four backend tiers — `matplotlib` (static), `interactive` (HoloViz),
`web` (MapLibre) and `3d` (PyVista). earthlens contributes only the parts that are
genuinely Earth Engine's — catalog-driven style hints and the two server-side
preview URLs — plus a `Visualizer` facade that calls a digitalearth verb.

One download, rendered four ways.

```bash
pip install earthlens[viz]                 # base: the static (matplotlib) tier
pip install 'digitalearth[web,interactive,3d]'   # the slippy-map and 3-D tiers
```

## Setup

In [ ]:
import os

os.environ.setdefault("PYVISTA_OFF_SCREEN", "true")  # render the 3-D tier headless

from pathlib import Path

import holoviews as hv
from IPython.display import Image

from earthlens.gee import GEE
from earthlens.gee.viz import style_for

hv.extension("bokeh")  # load the HoloViz JS so the interactive tier renders inline

OUT_DIR = Path("out") / "visualization"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SERVICE_ACCOUNT = os.environ["GEE_SERVICE_ACCOUNT"]
SERVICE_KEY = os.environ["GEE_SERVICE_KEY"]

## 1. Style hints from the catalog

`style_for` answers "how should this dataset be drawn?" from the catalog rather
than from a guess, and `source` says where the answer came from — the publisher's
own `gee:visualizations` rendering (894 of the 1104 curated datasets carry one),
or a heuristic over the band's declared range. No network needed.

In [ ]:
S2 = "COPERNICUS/S2_SR_HARMONIZED"
rgb = style_for(S2, ("B4", "B3", "B2"))
dem = style_for("USGS/SRTMGL1_003", ("elevation",))

print("Sentinel-2 true colour:", rgb.source, rgb.bands, "| rgb:", rgb.is_rgb)
print("SRTM elevation       :", dem.source, dem.bands, "| limits:", dem.limits)

## 2. One elevation download, three backends

A single SRTM tile over the Mont Blanc massif (500–4800 m of relief). The same
`DownloadResult` is drawn by three digitalearth tiers through `GEE.visualize`.
The facade spells the stretch each tier's own way — `vmin`/`vmax` for matplotlib,
`clim` for HoloViz and PyVista — so the one catalog style drives all of them.

In [ ]:
dem_gee = GEE(
    start="2000-02-11",
    end="2000-02-12",
    dataset="USGS/SRTMGL1_003",
    variables=["elevation"],
    aoi=[6.7, 45.7, 7.1, 46.0],
    cadence="raw",
    path=OUT_DIR,
    scale=150.0,
    reducer="mean",
    materialize=True,
)
dem_gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)
dem_viz = dem_gee.visualize(dem_gee.download(progress_bar=False))
print("style:", dem_viz.hints.source, dem_viz.hints.field_kwargs())

**Static (matplotlib)** — over a basemap, with coastlines.

In [ ]:
static_map = dem_viz.still(backend="matplotlib", basemap=True, coastlines=True)
static_map.fig

**3-D (PyVista)** — the same tile as a terrain surface. Rendered headless to a PNG.

In [ ]:
scene_3d = dem_viz.still(backend="3d")
scene_3d.save(str(OUT_DIR / "mont_blanc_3d.png"))
Image(filename=str(OUT_DIR / "mont_blanc_3d.png"))

**Interactive (HoloViz)** — pan and zoom; the widget renders inline.

In [ ]:
dem_viz.still(backend="interactive")

## 3. A true-colour composite

A monthly Sentinel-2 median over Cairo and the Nile. A three-band request is an
RGB composite, which `quickmap` cannot draw, so the facade routes it to the tier's
own `rgb_composite` with the catalog's per-band limits.

In [ ]:
s2_gee = GEE(
    start="2021-06-01",
    end="2021-06-30",
    dataset=S2,
    variables=["B4", "B3", "B2"],
    aoi=[31.0, 29.9, 31.3, 30.2],
    temporal_resolution="monthly",
    path=OUT_DIR,
    scale=100.0,
    reducer="median",
    materialize=True,
)
s2_gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)
s2_viz = s2_gee.visualize(s2_gee.download(progress_bar=False))
s2_viz.still(backend="matplotlib").fig

## 4. The data itself, as Earth Engine tiles on a slippy map

The showcase of the earthlens→digitalearth seam: earthlens resolves an Earth
Engine `getMapId` tile template, digitalearth draws it on a MapLibre web map.
Nothing is downloaded — the viewport decides which tiles Earth Engine renders, so
you can pan and zoom the **whole** Sentinel-2 archive. A standalone, shareable
HTML is written alongside; the widget also renders inline.

In [ ]:
import ee

region = ee.Geometry.Rectangle([31.0, 29.9, 31.3, 30.2])
true_colour = (
    ee.ImageCollection(S2)
    .filterBounds(region)
    .filterDate("2021-06-01", "2021-06-30")
    .median()
    .select(["B4", "B3", "B2"])
)

web_map = s2_viz.tile_map(true_colour, backend="web", name="Sentinel-2 true colour")
web_map.save(str(OUT_DIR / "s2_slippy_map.html"), offline=True)
print("standalone map:", OUT_DIR / "s2_slippy_map.html")
web_map

## 5. Server-side previews — no download

Two previews are rendered by Earth Engine itself, before any pixels move:
`thumbnail` wraps `getThumbURL` (a PNG URL Earth Engine renders on demand), and
`tiles` wraps `getMapId` to the `{z}/{x}/{y}` template the map above consumed.
Both are stretched by the catalog hints.

In [ ]:
thumb_url = s2_viz.thumbnail(true_colour, dimensions=400)
print("tile template:", s2_viz.tiles(true_colour).url.split("/tiles/")[-1])
Image(url=thumb_url)

## 6. A timelapse

`timelapse` stacks a multi-date download into one animation in a single call to
digitalearth's collection-aware `animate` — here monthly means of daily CHIRPS
precipitation across the 2021 Sahel wet season, the monsoon sweeping north from
May and receding by November.

In [ ]:
chirps_gee = GEE(
    start="2021-05-01",
    end="2021-11-01",
    dataset="UCSB-CHG/CHIRPS/DAILY",
    variables=["precipitation"],
    aoi=[0.0, 10.0, 10.0, 17.0],
    temporal_resolution="monthly",
    path=OUT_DIR,
    scale=10000.0,
    reducer="mean",
    materialize=True,
)
chirps_gee.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)
chirps_result = chirps_gee.download(progress_bar=False)
print("frames:", chirps_result.collection().time_length)

gif_path = OUT_DIR / "sahel_monsoon.gif"
chirps_gee.visualize(chirps_result).timelapse(gif_path, cmap="Blues")
Image(filename=str(gif_path))

## What lives where

| Concern | Package |
|---|---|
| Fetching, catalogs, request shaping | earthlens |
| `getThumbURL` / `getMapId`, catalog style hints | earthlens (`earthlens.gee.viz`) |
| Maps, animations, charts, the four backend tiers | digitalearth |
| Colour ramps, glyphs, the actual drawing | cleopatra |

## See also

- [Usage](../../reference/gee/usage.md) — the request shape and the object API.
- [Server-side computation](../../reference/gee/server-side-compute.md).